In [1]:
library(tidyverse)
library(data.table)

── Attaching packages ─────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.3     ✔ purrr   0.3.4
✔ tibble  3.1.2     ✔ dplyr   1.0.6
✔ tidyr   1.1.3     ✔ stringr 1.4.0
✔ readr   1.4.0     ✔ forcats 0.5.1

── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose




In [2]:
var_ranges <- read.csv("asd_path_var.csv")

In [3]:
head(var_ranges)

,refsnp_id,chr_name,chrom_start,chrom_end,allele,start_1mb,end_1mb,start_2mb,end_2mb,start_5mb,end_5mb,start_hmb,end_hmb
,<chr>,<chr>,<int>,<int>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,rs398122524,1,100011413,100011413,C/T,99011413,101011413,98011413,102011413,95011413,105011413,99511413,100511413
2,rs141952252,1,100017814,100017814,A/G,99017814,101017814,98017814,102017814,95017814,105017814,99517814,100517814
3,rs775151826,1,99993575,99993575,C/G/T,98993575,100993575,97993575,101993575,94993575,104993575,99493575,100493575
4,rs778502240,1,99999284,99999284,C/T,98999284,100999284,97999284,101999284,94999284,104999284,99499284,100499284
5,rs903783945,1,99993628,99993628,G/A/T,98993628,100993628,97993628,101993628,94993628,104993628,99493628,100493628
6,rs866803539,1,100015347,100015347,A/AA,99015347,101015347,98015347,102015347,95015347,105015347,99515347,100515347


In [4]:
length(unique(var_ranges$refsnp_id))

[1] 147

In [5]:
dim(var_ranges)

[1] 155  13

In [4]:
var_ranges$start_qmb <- var_ranges$chrom_start - 250000

In [5]:
var_ranges$end_qmb <- var_ranges$chrom_end + 250000

In [4]:
 #scores <- fread("/u/home/a/aflynnca/project-pasaniuc/projects/20241015_asd_pgs/20241024_asd_pgs/20241125_asd_test/PGS002453_hmPOS_GRCh38.txt")

In [6]:
scores <- fread("/u/home/a/aflynnca/project-pasaniuc/projects/20241015_asd_pgs/20241024_asd_pgs/PGS000327_hmPOS_GRCh38.txt")

In [7]:
head(scores)

rsID,chr_name,chr_position,effect_allele,other_allele,effect_weight,variant_description,hm_source,hm_rsID,hm_chr,hm_pos,hm_inferOtherAllele
<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<int>,<lgl>
rs113901932,8,100228085,A,G,-0.0598031,,ENSEMBL,rs113901932,8,99215857,NA
rs151150581,8,100422898,T,C,0.0483043,,ENSEMBL,rs151150581,8,99410670,NA
rs76228276,8,100638095,A,C,-0.0610991,,ENSEMBL,rs76228276,8,99625867,NA
rs13260744,8,100831260,A,G,-0.0456984,,ENSEMBL,rs13260744,8,99819032,NA
rs7832556,8,100923156,A,G,0.0259016,,ENSEMBL,rs7832556,8,99910928,NA
rs7830618,8,100926037,T,C,-0.0382004,,ENSEMBL,rs7830618,8,99913809,NA


In [8]:
table(scores$hm_source)


 ENSEMBL liftover  Unknown 
     843    34237        7 

In [9]:
dim(scores)

[1] 35087    12

In [10]:
subset_by_ranges <- function(scores, var_ranges, pos_col, start_col, end_col, chr_col) {
  # Check if positions in scores are within any of the ranges in var_ranges and chromosomes match
  in_range <- sapply(
    1:nrow(scores),  # Loop through each row in scores
    function(i) {
      # Find matching chromosomes in var_ranges for each position
      chr_match <- var_ranges[[chr_col]] == scores[[chr_col]][i]
      # Check if the position falls within the range for matching chromosomes
      any(chr_match & scores[[pos_col]][i] >= var_ranges[[start_col]] & scores[[pos_col]][i] <= var_ranges[[end_col]])
    }
  )
  
  # Subset and return the filtered scores
  return(scores[in_range, ])
}



In [11]:
# Call the function with chromosome matching
score_sub_var <- subset_by_ranges(
  scores = scores,
  var_ranges = var_ranges,
  pos_col = "chr_position",     # Column in scores with positions
  start_col = "chrom_start",    # Start column in var_ranges
  end_col = "chrom_end",       # End column in var_ranges
  chr_col = "chr_name"          # Column in both scores and var_ranges for chromosomes
)

In [12]:
dim(score_sub_var)

[1]  0 12

ERROR: Error in parse(text = x, srcfile = src): <text>:10:1: unexpected '}'
9:   return(scores[in_range])
10: }
    ^


In [11]:
subset_by_ranges <- function(scores, var_ranges, 
                             score_pos_col, score_chr_col, 
                             range_start_col, range_end_col, range_chr_col) {
  # Check if positions in scores are within any of the ranges in var_ranges and chromosomes match
  in_range <- sapply(
    1:nrow(scores),  # Loop through each row in scores
    function(i) {
      # Find matching chromosomes in var_ranges for each position
      chr_match <- var_ranges[[range_chr_col]] == scores[[score_chr_col]][i]
      # Check if the position falls within the range for matching chromosomes
      any(chr_match & 
          scores[[score_pos_col]][i] >= var_ranges[[range_start_col]] & 
          scores[[score_pos_col]][i] <= var_ranges[[range_end_col]])
    }
  )
  
  # Subset and return the filtered scores
  return(scores[in_range, ])
}


In [20]:
score_sub_var <- subset_by_ranges(
  scores = scores,
  var_ranges = var_ranges,
  score_pos_col = "hm_pos",  # Position column in scores
  score_chr_col = "hm_chr",      # Chromosome column in scores
  range_start_col = "start_pos",   # Start column in var_ranges
  range_end_col = "end_pos",       # End column in var_ranges
  range_chr_col = "chr_name"     # Chromosome column in var_ranges
)


In [15]:
dim(score_sub_var)

[1]  0 12

In [19]:
score_sub_5mb <- subset_by_ranges(
  scores = scores,
  var_ranges = var_ranges,
  score_pos_col = "hm_pos",  # Position column in scores
  score_chr_col = "hm_chr",      # Chromosome column in scores
  range_start_col = "start_5mb",   # Start column in var_ranges
  range_end_col = "end_5mb",       # End column in var_ranges
  range_chr_col = "chr_name"     # Chromosome column in var_ranges
)


In [22]:
dim(score_sub_5mb)

[1] 4592   12

In [23]:
head(score_sub_5mb)

rsID,chr_name,chr_position,effect_allele,other_allele,effect_weight,variant_description,hm_source,hm_rsID,hm_chr,hm_pos,hm_inferOtherAllele
<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<int>,<lgl>
,15,23697376,T,C,-0.0598986,,liftover,,15,23452229,NA
,15,23723805,A,C,-0.0375980,,liftover,,15,23478658,NA
,15,23770491,A,T,-0.0306035,,liftover,,15,23525344,NA
,15,23814372,A,G,0.0341985,,liftover,,15,23569225,NA
,15,23829308,A,C,0.0358979,,liftover,,15,23584161,NA
,15,23847508,A,G,-0.0471964,,liftover,,15,23602361,NA


In [15]:
dim(score_sub_5mb)

[1] 4684   12

In [25]:
score_sub_2mb <- subset_by_ranges(
  scores = scores,
  var_ranges = var_ranges,
  score_pos_col = "hm_pos",  # Position column in scores
  score_chr_col = "hm_chr",      # Chromosome column in scores
  range_start_col = "start_2mb",   # Start column in var_ranges
  range_end_col = "end_2mb",       # End column in var_ranges
  range_chr_col = "chr_name"     # Chromosome column in var_ranges
)

In [26]:
dim(score_sub_2mb)

[1] 1867   12

In [27]:

score_sub_1mb <- subset_by_ranges(
  scores = scores,
  var_ranges = var_ranges,
  score_pos_col = "hm_pos",  # Position column in scores
  score_chr_col = "hm_chr",      # Chromosome column in scores
  range_start_col = "start_1mb",   # Start column in var_ranges
  range_end_col = "end_1mb",       # End column in var_ranges
  range_chr_col = "chr_name"     # Chromosome column in var_ranges
)

In [28]:
dim(score_sub_1mb)

[1] 994  12

In [29]:

score_sub_hmb <- subset_by_ranges(
  scores = scores,
  var_ranges = var_ranges,
  score_pos_col = "hm_pos",  # Position column in scores
  score_chr_col = "hm_chr",      # Chromosome column in scores
  range_start_col = "start_hmb",   # Start column in var_ranges
  range_end_col = "end_hmb",       # End column in var_ranges
  range_chr_col = "chr_name"     # Chromosome column in var_ranges
)

In [30]:
dim(score_sub_hmb)

[1] 529  12

In [31]:

score_sub_qmb <- subset_by_ranges(
  scores = scores,
  var_ranges = var_ranges,
  score_pos_col = "hm_pos",  # Position column in scores
  score_chr_col = "hm_chr",      # Chromosome column in scores
  range_start_col = "start_qmb",   # Start column in var_ranges
  range_end_col = "end_qmb",       # End column in var_ranges
  range_chr_col = "chr_name"     # Chromosome column in var_ranges
)


In [32]:
dim(score_sub_qmb)

[1] 249  12

In [37]:
fwrite(score_sub_qmb, file = "asd_grove_qmb.txt", sep = "\t", quote = FALSE)

In [38]:
fwrite(score_sub_hmb, file = "asd_grove_hmb.txt", sep = "\t", quote = FALSE)

In [39]:
fwrite(score_sub_1mb, file = "asd_grove_1mb.txt", sep = "\t", quote = FALSE)

In [40]:
fwrite(score_sub_2mb, file = "asd_grove_2mb.txt", sep = "\t", quote = FALSE)

In [24]:
fwrite(score_sub_5mb, file = "asd_grove_5mb.txt", sep = "\t", quote = FALSE)

In [ ]:
# #genome_build=GRCh38
# added to top of file after creation